# ABO Adaptive RAG · Qwen3-VL-8B 公开体验

1. 登录 Kaggle，选择 **Copy & Edit** 建立自己的副本。
2. 在 **Session options** 启用 **Internet** 和 NVIDIA GPU（可选 T4 ×2 或可用的 L4），保留附加的 Qwen3-VL 8B Instruct 模型。
3. 运行下面的初始化单元，等到「ABO 商品助手已就绪」。首次启动会下载公开数据并建立 147,702 条商品索引。
4. 在本机启动 [ABO 网页](https://github.com/AaronYue77/ABO-Adaptive-RAG#打开网页问答台)。保持 Kaggle 会话运行，在 **Run → Kaggle Jupyter Server** 复制自己的 **VSCode Compatible URL**，粘贴到网页的「连接 Kaggle GPU」。这个地址包含当前会话凭据。

也可以在本 Notebook 新建单元直接用 `ask("给 iPhone 8 Plus 找个手机壳")`。问答日志保存在当前会话中。下面的验收单元只使用固定公开演示题。

[ABO 数据](https://amazon-berkeley-objects.s3.amazonaws.com/index.html) · [Qwen 底座](https://www.kaggle.com/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1) · [正式 LoRA](https://github.com/AaronYue77/ABO-Adaptive-RAG/releases/tag/qwen3-vl-8b-abo-lora-v1)

新版支持每轮多任务规划、需求与推荐理由取证、独立历史续聊和通道恢复。第一个单元在报告就绪前加载8B；更新代码后重新运行该单元会释放旧实例并载入当前版本。网页各段对话、Notebook与终端分别保存上下文。


In [ ]:
import runpy
import sys
from pathlib import Path
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen
from uuid import uuid4
from zipfile import ZipFile

revision = "efa6cdf18289e3bd0aca2945d180442cc98ecd66"
project = Path("/tmp/abo-adaptive-public") / revision
if not project.is_dir():
    project.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=project.parent) as staging:
        staging = Path(staging)
        bundle = staging / "source.zip"
        url = f"https://codeload.github.com/AaronYue77/ABO-Adaptive-RAG/zip/{revision}"
        with urlopen(url, timeout=120) as source, bundle.open("wb") as target:
            copyfileobj(source, target)
        with ZipFile(bundle) as archive:
            archive.extractall(staging)
        (staging / f"ABO-Adaptive-RAG-{revision}").replace(project)

_abo_source_tag = "efa6cdf18289e3bd0aca2945d180442cc98ecd66"
_abo_previous_chat = globals().get("chat")
if _abo_previous_chat is not None and globals().get("_abo_release_tag") != _abo_source_tag:
    _abo_previous_chat.close()
    del globals()["chat"]
    _service_file = getattr(sys.modules.get("app.abo.service"), "__file__", None)
    if _service_file:
        _old_root = Path(_service_file).resolve().parents[2]
        for _name, _module in list(sys.modules.items()):
            _file = getattr(_module, "__file__", None)
            if _file and Path(_file).resolve().is_relative_to(_old_root):
                sys.modules.pop(_name, None)
        sys.path[:] = [_path for _path in sys.path if not str(_path).startswith(str(_old_root))]

if "chat" not in globals():
    sys.path.insert(0, str(project))
    bootstrap = runpy.run_path(str(project / "scripts/kaggle_abo_public.py"))
    chat = bootstrap["start"]()
    _abo_release_tag = _abo_source_tag

_abo_notebook_thread = globals().get("_abo_notebook_thread") or uuid4().hex


def ask(question, *, details=False):
    from app.abo.cloud import cloud_request

    row = cloud_request(
        chat, {"thread_id": _abo_notebook_thread, "request_id": uuid4().hex, "question": question}
    )["row"]
    print(row["display_answer"])
    for label in dict.fromkeys(item["label"] for item in row["display_sources"]):
        print("来源", label)
    if details:
        return row

In [ ]:
_ = bootstrap["verify"](chat)